# V6.5 — Camber Zero-Disk Offload

Bản này dành cho trường hợp Kaggle đã đầy đến mức **không ghi nổi file vài KB**.

Khác biệt quan trọng:

- cell đầu **dọn đúng các cache / staging tạm** của các notebook V6.4/V6.5 trước;
- worker script được ghi vào `/dev/shm` (RAM filesystem), **không ghi `/kaggle/working`**;
- Kaggle chỉ upload 2 script nhỏ lên Camber Stash;
- mọi dataset lớn được download, parse, package và upload trên **Camber compute**;
- worker External xử lý từng source tuần tự rồi xóa temp;
- Open Images xử lý **từng class một** để không giữ 5 class cùng lúc;
- parent background worker chỉ dùng **V6.2E base TRAIN** làm background pool, không tải overlay nếu không cần;
- không extract full V6.2E.

Nếu `camber.mpi` không có trong Kaggle kernel, notebook vẫn upload worker thành công và in đúng 2 yêu cầu để chạy bằng Camber Nova.

> Không train model trong notebook này. Đây chỉ là data staging/offload.

In [ ]:
# ==============================================================================
# 0. EMERGENCY CLEANUP — FREE KAGGLE DISK FIRST
# ==============================================================================
from pathlib import Path
import os, shutil, subprocess, sys, json, textwrap, hashlib

def disk(path="/kaggle/working"):
    total, used, free = shutil.disk_usage(path)
    print(
        f"{path}: total={total/1024**3:.2f} GB | "
        f"used={used/1024**3:.2f} GB | free={free/1024**3:.2f} GB"
    )
    return free

print("BEFORE CLEANUP")
disk()

# Chỉ xóa transient caches / failed reconstruction trees do các notebook vừa tạo.
# Không xóa notebook input hay Camber Stash.
cleanup_targets = [
    Path("/kaggle/working/v6p5_external_expansion"),
    Path("/kaggle/working/v65_camber_workers"),
    Path("/kaggle/working/v6p4_camber_bootstrap/cache"),
    Path("/kaggle/working/v6p4_camber_bootstrap/v6p2e_runtime"),
    Path("/tmp/v6p4_clean_runtime"),
    Path("/tmp/v6p4_postclean_bootstrap_cache"),
    Path("/tmp/v6p5_external_expansion"),
    Path("/tmp/v65_external_worker"),
    Path("/tmp/v65_parent_bg_worker"),
    Path.home() / ".cache" / "huggingface" / "hub",
    Path.home() / ".cache" / "pip",
]

for p in cleanup_targets:
    try:
        if p.exists():
            print("remove:", p)
            if p.is_dir():
                shutil.rmtree(p, ignore_errors=True)
            else:
                p.unlink(missing_ok=True)
    except Exception as e:
        print("WARN cleanup:", p, repr(e))

# Best-effort apt/pip temp cleanup.
for p in [Path("/tmp/pip-build-tracker"), Path("/tmp/pip-install")]:
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)

print("\nAFTER CLEANUP")
free = disk()

# Worker scripts will still use RAM, so we only need a little disk headroom for CLI/runtime.
if free < 200 * 1024**2:
    print(
        "WARN: còn dưới 200 MB. Worker scripts vẫn ghi /dev/shm, "
        "nhưng nên Restart Session nếu môi trường tiếp tục bất ổn."
    )

RAM_TMP = Path("/dev/shm/v65_camber_offload")
RAM_TMP.mkdir(parents=True, exist_ok=True)
print("RAM tmp:", RAM_TMP)

In [ ]:
# ==============================================================================
# 1. CAMBER AUTH — NO DATA DOWNLOAD ON KAGGLE
# ==============================================================================
STAGE_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
JOB_ROOT = f"{STAGE_ROOT}/jobs"

key = os.environ.get("CAMBER_API_KEY")
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("CAMBER_API_KEY")
    except Exception:
        key = None

if not key:
    raise RuntimeError("Bật CAMBER_API_KEY trong Kaggle Secrets.")

os.environ["CAMBER_API_KEY"] = key

camber_bin = Path.home() / ".camber/bin/camber"

# Thường Camber CLI đã có từ các run trước.
# Chỉ cài nếu thiếu, sau cleanup lúc này đã có chút headroom.
if shutil.which("camber") is None and not camber_bin.exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True,
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:"
    + os.environ.get("PATH", "")
)
CAMBER = shutil.which("camber") or str(camber_bin)

assert Path(CAMBER).exists(), "Camber CLI not found"

r = subprocess.run(
    [CAMBER, "me"],
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)
print(r.stdout or r.stderr)

In [ ]:
# ==============================================================================
# 2. WORKER A — EXTERNAL SOURCES, SEQUENTIAL / LOW LOCAL DISK
# ==============================================================================
EXTERNAL_WORKER = r"""
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile, tarfile, xml.etree.ElementTree as ET

ROOT = Path("/tmp/v65_external_worker")
ROOT.mkdir(parents=True, exist_ok=True)

STAGE_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
OI_CLASSES = ["Balloon", "Kite", "Plastic bag", "Bird", "Person"]
OI_MAX_PER_CLASS = 160
SEED = 20261007
MPCD_GDRIVE_ID = "1KyciMmwL2_p_-mwckHFqG5fkBA1jzATv"
HF_REPO = "sxiong/Power-equipment-image-dataset"
HF_FILE = "transmission line object detection.zip"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "pillow", "pyyaml", "gdown",
    "huggingface_hub", "fiftyone==1.22.1"
])

import pandas as pd
import yaml, gdown
from PIL import Image
from huggingface_hub import hf_hub_download
import fiftyone.utils.openimages as fouo

def ensure_camber():
    c = shutil.which("camber") or str(Path.home()/".camber/bin/camber")
    if not Path(c).exists():
        subprocess.check_call(
            "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
            shell=True,
        )
        os.environ["PATH"] = f"{Path.home()}/.camber/bin:" + os.environ.get("PATH","")
        c = shutil.which("camber") or str(Path.home()/".camber/bin/camber")
    assert Path(c).exists()
    return c

CAMBER = ensure_camber()

def run(cmd):
    r = subprocess.run(cmd, text=True, capture_output=True, env=os.environ.copy())
    if r.returncode != 0:
        raise RuntimeError((r.stderr or r.stdout)[:8000])
    return r

def put(local, remote):
    run([CAMBER, "stash", "cp", str(local), remote])

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            b = f.read(1024*1024)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def norm_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x:int(x))]
    return list(names or [])

def read_yolo(lp):
    rows=[]
    if not Path(lp).exists():
        return rows
    for line in Path(lp).read_text(errors="ignore").splitlines():
        p=line.split()
        if len(p)<5:
            continue
        try:
            rows.append((int(float(p[0])), *map(float,p[1:5])))
        except Exception:
            pass
    return rows

def write_yolo(lp, rows):
    Path(lp).parent.mkdir(parents=True, exist_ok=True)
    txt="\n".join(
        f"{int(c)} {x:.8f} {y:.8f} {w:.8f} {h:.8f}"
        for c,x,y,w,h in rows if w>0 and h>0
    )
    Path(lp).write_text(txt + ("\n" if txt else ""))

def xyxy_to_yolo(box):
    x1,y1,x2,y2=box
    return [(x1+x2)/2,(y1+y2)/2,max(0,x2-x1),max(0,y2-y1)]

source_manifest = []

# =====================================================================
# A. OPEN IMAGES: ONE CLASS AT A TIME
# =====================================================================
boxable=set(fouo.get_classes(version="v7"))
segable=set(fouo.get_segmentation_classes(version="v7"))

for cls in OI_CLASSES:
    work = ROOT / ("oi_" + cls.lower().replace(" ","_"))
    if work.exists():
        shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True, exist_ok=True)

    labels=["detections"]
    if cls in segable:
        labels.append("segmentations")

    fouo.download_open_images_split(
        dataset_dir=str(work),
        split="train",
        version="v7",
        label_types=labels,
        classes=[cls],
        shuffle=True,
        seed=SEED,
        max_samples=OI_MAX_PER_CLASS,
        num_workers=8,
    )

    tar_path = ROOT / f"open_images_v7_{cls.lower().replace(' ','_')}.tar.gz"
    with tarfile.open(tar_path, "w:gz") as t:
        t.add(work, arcname=work.name)

    remote = f"{STAGE_ROOT}/open_images/{tar_path.name}"
    put(tar_path, remote)

    source_manifest.append({
        "source":"OpenImagesV7",
        "split":"train",
        "class":cls,
        "role":"foreign_object_appearance_source",
        "archive":remote,
    })

    shutil.rmtree(work, ignore_errors=True)
    tar_path.unlink(missing_ok=True)

# =====================================================================
# B. MPCD — PROCESS THEN DELETE
# =====================================================================
mpcd_zip=ROOT/"MPCD.zip"
mpcd_dir=ROOT/"MPCD"

result=gdown.download(
    f"https://drive.google.com/uc?id={MPCD_GDRIVE_ID}",
    str(mpcd_zip),
    quiet=False,
)
if result is None:
    raise RuntimeError("MPCD download failed")

mpcd_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(mpcd_zip) as z:
    z.extractall(mpcd_dir)
mpcd_zip.unlink(missing_ok=True)

stage=ROOT/"mpcd_stage"
for s in ["train","external_eval"]:
    (stage/"images"/s).mkdir(parents=True, exist_ok=True)
    (stage/"labels"/s).mkdir(parents=True, exist_ok=True)

rows_meta=[]

for yp in mpcd_dir.rglob("data.yaml"):
    try:
        cfg=yaml.safe_load(yp.read_text())
        names=norm_names(cfg.get("names"))
    except Exception:
        continue

    broken_ids=[]
    for cid,n in enumerate(names):
        z=str(n).lower().replace("_"," ").replace("-"," ")
        if any(k in z for k in ["broken","fracture","fractured","discontinu","strand break"]):
            broken_ids.append(cid)

    if not broken_ids:
        continue

    base=Path(cfg.get("path",yp.parent))
    if not base.is_absolute():
        base=(yp.parent/base).resolve()

    for sk in ["train","val","test"]:
        raw=cfg.get(sk)
        if raw is None:
            continue

        p=Path(raw)
        if not p.is_absolute():
            p=base/p

        if p.is_file():
            imgs=[]
            for x in p.read_text().splitlines():
                if not x.strip():
                    continue
                q=Path(x.strip())
                imgs.append(q if q.is_absolute() else base/q)
        elif p.exists():
            imgs=[x for x in p.rglob("*") if x.suffix.lower() in {".jpg",".jpeg",".png",".webp",".bmp"}]
        else:
            continue

        for ip in imgs:
            s=str(ip).replace("\\","/")
            if "/images/" not in s:
                continue

            lp=Path(s.replace("/images/","/labels/")).with_suffix(".txt")
            if not lp.exists():
                continue

            mapped=[(5,x,y,w,h) for c,x,y,w,h in read_yolo(lp) if c in broken_ids]
            if not mapped:
                continue

            target="train" if sk=="train" else "external_eval"
            sha=sha256_file(ip)
            di=stage/"images"/target/f"mpcd_{sha[:16]}{ip.suffix.lower()}"
            dl=stage/"labels"/target/f"mpcd_{sha[:16]}.txt"
            shutil.copy2(ip,di)
            write_yolo(dl,mapped)

            rows_meta.append({
                "source_split":sk,
                "target_split":target,
                "source_sha256":sha,
                "n_boxes":len(mapped),
            })

pd.DataFrame(rows_meta).to_csv(stage/"manifest.csv",index=False)

tar_path=ROOT/"mpcd_broken_strand_stage.tar.gz"
with tarfile.open(tar_path,"w:gz") as t:
    t.add(stage,arcname="mpcd_broken_strand")

remote=f"{STAGE_ROOT}/mpcd/{tar_path.name}"
put(tar_path,remote)

source_manifest.append({
    "source":"MPCD",
    "role":"broken_strand_direct_powerline",
    "mapped_images":len(rows_meta),
    "archive":remote,
})

shutil.rmtree(mpcd_dir,ignore_errors=True)
shutil.rmtree(stage,ignore_errors=True)
tar_path.unlink(missing_ok=True)

# =====================================================================
# C. POWER EQUIPMENT — PROCESS THEN DELETE
# =====================================================================
hf_cache=ROOT/"hf_cache"
hf_path=Path(hf_hub_download(
    repo_id=HF_REPO,
    filename=HF_FILE,
    repo_type="dataset",
    cache_dir=str(hf_cache),
))

pe_dir=ROOT/"power_equipment"
pe_dir.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(hf_path) as z:
    z.extractall(pe_dir)
shutil.rmtree(hf_cache,ignore_errors=True)

stage=ROOT/"power_equipment_stage"
for s in ["train","external_eval"]:
    (stage/"images"/s).mkdir(parents=True,exist_ok=True)
    (stage/"labels"/s).mkdir(parents=True,exist_ok=True)

imgs={}
for p in pe_dir.rglob("*"):
    if p.suffix.lower() in {".jpg",".jpeg",".png",".webp",".bmp"}:
        imgs.setdefault(p.stem,[]).append(p)

mapping={"dx_dg":5,"yw":4,"nw":3}
review_only={"dx_sg":"strand_loosening_review_only"}
meta=[]
review=[]

for xp in pe_dir.rglob("*.xml"):
    try:
        root=ET.parse(xp).getroot()
    except Exception:
        continue

    filename=root.findtext("filename")
    ip=None
    if filename:
        q=xp.parent/filename
        if q.exists():
            ip=q
        elif imgs.get(Path(filename).stem):
            ip=imgs[Path(filename).stem][0]
    elif imgs.get(xp.stem):
        ip=imgs[xp.stem][0]

    if ip is None:
        continue

    size=root.find("size")
    if size is None:
        with Image.open(ip) as im:
            W,H=im.size
    else:
        W=int(float(size.findtext("width")))
        H=int(float(size.findtext("height")))

    mapped=[]

    for obj in root.findall("object"):
        name=(obj.findtext("name") or "").strip()
        bb=obj.find("bndbox")
        if bb is None:
            continue

        box=[
            max(0,min(1,float(bb.findtext("xmin"))/W)),
            max(0,min(1,float(bb.findtext("ymin"))/H)),
            max(0,min(1,float(bb.findtext("xmax"))/W)),
            max(0,min(1,float(bb.findtext("ymax"))/H)),
        ]

        if name in mapping:
            xc,yc,w,h=xyxy_to_yolo(box)
            mapped.append((mapping[name],xc,yc,w,h))

        if name in review_only:
            review.append({
                "source_sha256":sha256_file(ip),
                "source_class":name,
                "role":review_only[name],
                "bbox":json.dumps(box),
            })

    if not mapped:
        continue

    sha=sha256_file(ip)
    frac=int(hashlib.sha256(sha.encode()).hexdigest()[:12],16)/float(16**12)
    target="train" if frac<0.80 else "external_eval"

    di=stage/"images"/target/f"pe_{sha[:16]}{ip.suffix.lower()}"
    dl=stage/"labels"/target/f"pe_{sha[:16]}.txt"
    shutil.copy2(ip,di)
    write_yolo(dl,mapped)

    meta.append({
        "source_sha256":sha,
        "target_split":target,
        "mapped_classes":",".join(map(str,sorted(set(x[0] for x in mapped)))),
        "n_boxes":len(mapped),
    })

pd.DataFrame(meta).to_csv(stage/"manifest.csv",index=False)
pd.DataFrame(review).to_csv(stage/"dx_sg_review_only.csv",index=False)

tar_path=ROOT/"power_equipment_stage.tar.gz"
with tarfile.open(tar_path,"w:gz") as t:
    t.add(stage,arcname="power_equipment_stage")

remote=f"{STAGE_ROOT}/power_equipment/{tar_path.name}"
put(tar_path,remote)

source_manifest.append({
    "source":"PowerEquipment",
    "role":"direct_powerline",
    "mapping":{
        "dx_dg":"broken_strand",
        "yw":"foreign_object",
        "nw":"bird_nest",
        "dx_sg":"review_only",
    },
    "mapped_images":len(meta),
    "archive":remote,
})

shutil.rmtree(pe_dir,ignore_errors=True)
shutil.rmtree(stage,ignore_errors=True)
tar_path.unlink(missing_ok=True)

# =====================================================================
# FINAL MANIFEST
# =====================================================================
manifest_path=ROOT/"external_stage_manifest.json"
manifest_path.write_text(json.dumps({
    "stage":"v6p5_external_sources",
    "sources":source_manifest,
    "policy":{
        "OpenImages_full_images_direct_merge":False,
        "OpenImages_split":"train",
        "MPCD_train_to_train":True,
        "MPCD_val_test_to_external_eval":True,
        "PowerEquipment_dx_sg_direct_merge":False,
        "test_modified":False,
    },
},indent=2))

put(manifest_path,f"{STAGE_ROOT}/external_stage_manifest.json")

shutil.rmtree(ROOT,ignore_errors=True)
print("DONE external worker")
"""
print("External worker prepared:", len(EXTERNAL_WORKER), "chars")

In [ ]:
# ==============================================================================
# 3. WORKER B — SMALL TRAIN BACKGROUND POOL, BASE TAR ONLY
# ==============================================================================
PARENT_WORKER = r"""
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile, tarfile
import pandas as pd

subprocess.check_call([
    sys.executable,"-m","pip","install","-q","pandas"
])

ROOT=Path("/tmp/v65_parent_bg_worker")
CACHE=ROOT/"cache"
POOL=ROOT/"pool"

CACHE.mkdir(parents=True,exist_ok=True)
(POOL/"images").mkdir(parents=True,exist_ok=True)
(POOL/"labels").mkdir(parents=True,exist_ok=True)

V6ROOT="stash://bunpmc/projects/yolov11sdi/datasets"
TAG="v6p2e_23562a45b343"
STAGE="stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
MAX_BG=700
SEED="20261007"

def ensure_camber():
    c=shutil.which("camber") or str(Path.home()/".camber/bin/camber")
    if not Path(c).exists():
        subprocess.check_call(
            "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
            shell=True,
        )
        os.environ["PATH"]=f"{Path.home()}/.camber/bin:"+os.environ.get("PATH","")
        c=shutil.which("camber") or str(Path.home()/".camber/bin/camber")
    assert Path(c).exists()
    return c

CAMBER=ensure_camber()

def run(cmd):
    r=subprocess.run(cmd,text=True,capture_output=True,env=os.environ.copy())
    if r.returncode!=0:
        raise RuntimeError((r.stderr or r.stdout)[:8000])
    return r

def get(remote,local):
    local=Path(local)
    local.parent.mkdir(parents=True,exist_ok=True)
    if not local.exists():
        run([CAMBER,"stash","cp",remote,str(local)])
    return local

def put(local,remote):
    run([CAMBER,"stash","cp",str(local),remote])

freeze_path=get(
    f"{V6ROOT}/v6p2e/{TAG}/v6p2e.freeze.json",
    CACHE/"freeze.json",
)
freeze=json.loads(freeze_path.read_text())

# Intentionally use BASE source only for synthetic backgrounds.
# No need to download overlay archive.
base_tar=get(
    f"{V6ROOT}/curated_v6_rs1280_{freeze['base_tag']}.tar",
    CACHE/"base.tar",
)
labels_zip=get(
    f"{V6ROOT}/v6p2e/{TAG}/v6p2e_reconciled_labels.zip",
    CACHE/"labels.zip",
)
manifest_csv=get(
    f"{V6ROOT}/v6p2e/{TAG}/v6p2e_split_manifest.csv",
    CACHE/"manifest.csv",
)

manifest=pd.read_csv(manifest_csv)

tf=tarfile.open(base_tar,"r:*")
rz=zipfile.ZipFile(labels_zip,"r")

def build_idx(tf):
    idx={}
    for m in tf.getmembers():
        if not m.isfile():
            continue
        n=m.name.replace("\\","/").lstrip("./")
        pi=n.find("images/")
        pl=n.find("labels/")
        key=n[pi:] if pi>=0 else (n[pl:] if pl>=0 else None)
        if key is not None and key not in idx:
            idx[key]=m
    return idx

idx=build_idx(tf)
ridx={Path(n).name:n for n in rz.namelist() if n.endswith(".txt")}

def keys(row):
    old=str(row["image_path_source"]).replace("\\","/")
    split=str(row["previous_v6p1_split"])
    marker=f"/images/{split}/"
    suffix=old.split(marker,1)[1]
    if not suffix.startswith("base/"):
        return None
    rel=suffix[len("base/"):]
    return (
        rel,
        f"images/{split}/{rel}",
        f"labels/{split}/{Path(rel).with_suffix('.txt').as_posix()}",
    )

def label_text(row, lblk):
    rn=f"{row['sha256']}.txt"
    if str(row.get("resolution","")).startswith("merged") and rn in ridx:
        return rz.read(ridx[rn]).decode("utf-8",errors="ignore")
    m=idx.get(lblk)
    if m is None:
        raise KeyError(lblk)
    return tf.extractfile(m).read().decode("utf-8",errors="ignore")

def has_insulator(txt):
    for line in txt.splitlines():
        p=line.split()
        if len(p)>=5:
            try:
                if int(float(p[0]))==0:
                    return True
            except Exception:
                pass
    return False

tr=manifest[
    (manifest["split"].astype(str)=="train")
    & manifest["image_path_source"].astype(str).str.contains("/base/")
].copy()

tr["_order"]=tr["sha256"].astype(str).map(
    lambda x: hashlib.sha256((x+SEED).encode()).hexdigest()
)
tr=tr.sort_values("_order")

rows=[]

for i,row in tr.iterrows():
    if len(rows)>=MAX_BG:
        break

    k=keys(row)
    if k is None:
        continue

    rel,imgk,lblk=k

    try:
        txt=label_text(row,lblk)
        if not has_insulator(txt):
            continue

        m=idx.get(imgk)
        if m is None:
            continue

        raw=tf.extractfile(m).read()
        suffix=Path(rel).suffix.lower() or ".jpg"
        name=f"{str(row['sha256'])[:16]}_{i:06d}{suffix}"

        (POOL/"images"/name).write_bytes(raw)
        (POOL/"labels"/Path(name).with_suffix(".txt")).write_text(txt)

        rows.append({
            "sha256":str(row["sha256"]),
            "manifest_index":int(i),
            "source":"V6.2E-base-train",
            "image":f"images/{name}",
            "label":f"labels/{Path(name).with_suffix('.txt').name}",
        })
    except Exception:
        continue

tf.close()
rz.close()

pd.DataFrame(rows).to_csv(POOL/"manifest.csv",index=False)

assert len(rows)>=300, f"Too few backgrounds selected: {len(rows)}"

out=ROOT/"v6p2e_base_train_background_pool.tar.gz"
with tarfile.open(out,"w:gz") as t:
    t.add(POOL,arcname="v6p2e_base_train_background_pool")

remote=f"{STAGE}/parent_background_pool/{out.name}"
put(out,remote)

meta=ROOT/"parent_background_pool.json"
meta.write_text(json.dumps({
    "source_dataset":"V6.2E",
    "source_tag":TAG,
    "source_subset":"base provenance only",
    "split":"train",
    "selection":"contains class-0 insulator",
    "count":len(rows),
    "archive":remote,
},indent=2))

put(meta,f"{STAGE}/parent_background_pool/{meta.name}")

shutil.rmtree(ROOT,ignore_errors=True)
print("DONE parent background worker")
"""
print("Parent worker prepared:", len(PARENT_WORKER), "chars")

In [ ]:
# ==============================================================================
# 4. WRITE WORKERS TO RAM (/dev/shm), UPLOAD TO CAMBER, THEN DELETE RAM FILES
# ==============================================================================
external_py = RAM_TMP / "v65_external_worker.py"
parent_py = RAM_TMP / "v65_parent_bg_worker.py"

# Compile before writing/upload.
compile(EXTERNAL_WORKER, str(external_py), "exec")
compile(PARENT_WORKER, str(parent_py), "exec")

external_py.write_text(EXTERNAL_WORKER)
parent_py.write_text(PARENT_WORKER)

print("RAM worker files:")
print(external_py, external_py.stat().st_size)
print(parent_py, parent_py.stat().st_size)

def stash_put(local_path, remote):
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(local_path), remote],
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    if r.returncode != 0:
        raise RuntimeError(r.stderr or r.stdout)
    print("uploaded:", remote)

REMOTE_EXTERNAL = f"{JOB_ROOT}/{external_py.name}"
REMOTE_PARENT = f"{JOB_ROOT}/{parent_py.name}"

stash_put(external_py, REMOTE_EXTERNAL)
stash_put(parent_py, REMOTE_PARENT)

# Remove even the RAM copies after upload.
external_py.unlink(missing_ok=True)
parent_py.unlink(missing_ok=True)

print("\nWorkers on Camber:")
print(" ", REMOTE_EXTERNAL)
print(" ", REMOTE_PARENT)

In [ ]:
# ==============================================================================
# 5. SUBMIT CAMBER CPU JOBS IF SDK IS AVAILABLE
# Current Camber docs expose camber.mpi.create_job(...)
# Use MEDIUM CPU nodes for headroom with archive indexing.
# ==============================================================================
NODE_SIZE = "MEDIUM"

commands = {
    "external": (
        "bash -lc 'mkdir -p /tmp/v65job && "
        f"camber stash cp {REMOTE_EXTERNAL} /tmp/v65job/external.py && "
        "python /tmp/v65job/external.py'"
    ),
    "parent_background": (
        "bash -lc 'mkdir -p /tmp/v65job && "
        f"camber stash cp {REMOTE_PARENT} /tmp/v65job/parent.py && "
        "python /tmp/v65job/parent.py'"
    ),
}

submitted = {}

try:
    import camber

    if not hasattr(camber, "mpi"):
        raise ImportError("camber.mpi unavailable in this Kaggle kernel")

    for name, cmd in commands.items():
        print("Submitting", name)
        job = camber.mpi.create_job(
            command=cmd,
            node_size=NODE_SIZE,
            num_nodes=1,
            with_gpu=False,
        )
        submitted[name] = str(job)
        print(" ->", job)

except Exception as e:
    print("Automatic submit unavailable:", repr(e))
    print()
    print("Workers ARE already stored on Camber Stash.")
    print("Use Camber Nova with these two requests:")
    print()
    print(
        f"Run {REMOTE_EXTERNAL} as a Python script on one {NODE_SIZE} CPU node. "
        "It stages Open Images, MPCD, and Power Equipment into "
        f"{STAGE_ROOT} and cleans local temp files as it goes."
    )
    print()
    print(
        f"Run {REMOTE_PARENT} as a Python script on one {NODE_SIZE} CPU node. "
        "It builds a small V6.2E BASE TRAIN background pool and writes it into "
        f"{STAGE_ROOT}/parent_background_pool."
    )

print("\nJob commands:")
print(json.dumps(commands, indent=2))

In [ ]:
# ==============================================================================
# 6. SERVER-SIDE STATUS — DOES NOT DOWNLOAD DATA TO KAGGLE
# ==============================================================================
targets = [
    STAGE_ROOT,
    f"{STAGE_ROOT}/jobs",
    f"{STAGE_ROOT}/open_images",
    f"{STAGE_ROOT}/mpcd",
    f"{STAGE_ROOT}/power_equipment",
    f"{STAGE_ROOT}/parent_background_pool",
]

for target in targets:
    print("\n###", target)
    r = subprocess.run(
        [CAMBER, "stash", "ls", "-r", target],
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    print((r.stdout or r.stderr)[:15000])

print("\nKaggle disk after offload launcher:")
disk()

## Kết quả mong đợi trên Camber Stash

Root:

`stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/`

Các artifact chính:

- `open_images/open_images_v7_balloon.tar.gz`
- `open_images/open_images_v7_kite.tar.gz`
- `open_images/open_images_v7_plastic_bag.tar.gz`
- `open_images/open_images_v7_bird.tar.gz`
- `open_images/open_images_v7_person.tar.gz`
- `mpcd/mpcd_broken_strand_stage.tar.gz`
- `power_equipment/power_equipment_stage.tar.gz`
- `parent_background_pool/v6p2e_base_train_background_pool.tar.gz`
- `external_stage_manifest.json`

Sau bước này, synthetic composition + QA tiếp theo cũng nên chạy trên Camber, không kéo các archive trên về Kaggle.

Kaggle chỉ cần đọc/download CSV/JSON/contact-sheet nhỏ sau cùng.